# 16 · Ficha de probabilidad desde pronóstico, por plazo

> **No es una alerta ni una predicción para decidir evacuaciones.** Es la frecuencia histórica de días con huaico *catalogado* bajo pronósticos parecidos. Ante una emergencia llama al 115 (INDECI).

**Pregunta.** Si en lugar de la lluvia observada solo tenemos el pronóstico a 24, 48 o 72 horas, ¿qué probabilidad de un día con huaico catalogado podemos dar, con qué intervalo y qué tan fiable es?

**Datos.** Las probabilidades fuera de muestra del backtest con el reforecast GEFSv12 2000-2019 (capítulo 19): 2 952 días de temporada por plazo, 55 con huaico (32 sin 2017). `pp_qm` es el modelo del capítulo 15 entrenado con ERA5 y alimentado con la lluvia pronosticada corregida por cuantiles.

**Qué se prueba (escrito antes de leer las tablas de este cuaderno; las del capítulo 19 ya se conocían):**
- **H1** una recalibración de Platt por plazo, ajustada dejando una temporada fuera, acerca la razón observado/predicho a 1.
- **H2** esa recalibración mejora el Brier y el log-loss en los tres plazos, con y sin 2017.
- **H3** las bandas de probabilidad altas son fiables en los tres plazos.
- **H4** el resultado no depende de 2017.
- **Regla de reporte (decidida al ver que la curva de Platt se sale en la cola):** la ficha informa la frecuencia observada de la banda con su intervalo; la curva queda como dato del modelo.

In [ ]:
from pathlib import Path

import pandas as pd

from illapa_research import ficha_pronostico as F

RAIZ = Path("..") if Path("../catalogo").exists() else Path(".")
R = F.correr(RAIZ, n_boot=500)
oof = R["oof"]
print(oof.groupby("plazo").agg(dias=("huaico", "size"), huaicos=("huaico", "sum")))

## 1. Calibración por plazo (H1 y H2)

Cada fila compara la probabilidad cruda con la calibrada dejando una temporada fuera. «sin 2017» ajusta y evalúa sin ese año. `brier_skill` > 0 es mejor que la climatología por mes; `obs_sobre_pred` = 1 es calibración perfecta.

In [ ]:
R["calibracion"].round(4)

## 2. Bandas de fiabilidad por plazo (H3 y H4)

De los días cuya probabilidad calibrada cayó en cada banda, cuántos tuvieron huaico (intervalo de Wilson del 80 %). Es la lectura más directa de «qué tan fiable es esta cifra».

In [ ]:
R["bandas"].round(3)[["plazo", "subconjunto", "banda", "dias", "huaicos", "observado", "ic80_bajo", "ic80_alto", "predicho_medio"]]

## 3. Tres fichas de ejemplo

Se toman días reales del backtest (con la probabilidad cruda que tuvieron, sin mirar si hubo huaico) y se arma la ficha como se armaría para un pronóstico nuevo.

In [ ]:
m = R["modelo"]
casos = []
d1 = oof[oof.plazo == 1].sort_values("pp_qm")
for etiqueta, fila in (("el día de mayor probabilidad a 24 h", d1.iloc[-1]),
                       ("un día de probabilidad media a 24 h", d1.iloc[int(len(d1) * 0.97)]),
                       ("un día de probabilidad alta a 48 h", oof[oof.plazo == 2].sort_values("pp_qm").iloc[-5])):
    f = m.ficha(float(fila["pp_qm"]), int(fila["plazo"]))
    casos.append({"caso": etiqueta, "dia": fila["dia"].date(), "plazo": f["plazo"], "p_cruda": round(f["probabilidad_cruda"], 3),
                  "probabilidad": round(f["probabilidad"], 3), "ic80": tuple(round(v, 3) for v in f["intervalo_80"]),
                  "curva_calibrada": round(f["probabilidad_modelo_calibrada"], 3), "sin_2017": round(f["probabilidad_sin_2017"], 3), "veces_la_base": round(f["veces_la_base"], 1),
                  "banda": f["banda_de_fiabilidad"]["banda"], "dias_banda": f["banda_de_fiabilidad"]["dias"],
                  "huaicos_banda": f["banda_de_fiabilidad"]["huaicos"], "hubo_huaico": int(fila["huaico"])})
pd.DataFrame(casos)

In [ ]:
for c in casos:
    f = m.ficha(c["p_cruda"], int(c["plazo"][:2]) // 24)
    print(c["caso"], "->", f["advertencias"] or "sin advertencias")
print()
print(f["limites"])

## Lectura

- **A 24 h** la probabilidad cruda conserva más habilidad en Brier, pero exagera (observado/predicho ≈ 0,78); la calibración corrige el nivel y cuesta un poco de Brier. La banda de 25 % o más tuvo huaico en 41 % de los días con 2017 y en 20 % sin él.
- **A 48 y 72 h** la calibración lleva el Brier skill de negativo a casi cero: **la cifra no es mejor que la climatología**; el valor que queda es ordenar los días. Las bandas altas **no son fiables** (15 % y 8 % observado frente a 45 % y 37 % predicho).
- **2017 pesa en todo**: sin ese año las frecuencias de las bandas altas se reducen a la mitad o menos.
- **La cifra que se informa es la frecuencia observada en la banda**, con su intervalo de Wilson, no la curva de Platt: en la cola la curva sube a 90 % (24 h, probabilidad cruda de 0,99) cuando de los 27 días históricos de esa banda solo el 41 % tuvo huaico. Platt se conserva como `probabilidad_modelo_calibrada` y la ficha avisa cuando se separan.
- Para el uso: dar la frecuencia observada de la banda con su intervalo y las advertencias, no una cifra única.